# Fake News Detection — Training & Evaluation

**Project:** Fake News Detector for Students (AICTE)

This notebook builds and rigorously evaluates the machine-learning half of the hybrid
system: a **TF-IDF (word + character n-grams) → Logistic Regression** classifier that
estimates whether an article is fake or real. The trained model is reused by the CLI
(`cli.py`) and the Streamlit web app (`app.py`).

Because the app reports a **credibility score**, we don't stop at accuracy — we also
check **cross-validation stability**, **ROC-AUC**, and **probability calibration**
(a score of "80% real" should be right ~80% of the time).

> **Data:** the default is a synthetic demo dataset (with borderline cases + label noise
> so metrics are realistic). For publishable accuracy, drop the Kaggle *Fake and Real News*
> `Fake.csv` / `True.csv` into `data/` and re-run `data/make_dataset.py`.

## 1. Setup

In [ ]:
import sys, os, subprocess
sys.path.insert(0, "src")   # run this notebook from the project root
print(subprocess.run([sys.executable, "data/make_dataset.py"], capture_output=True, text=True).stdout)

## 2. Load and inspect

In [ ]:
import pandas as pd
df = pd.read_csv("data/news.csv")
print("rows:", len(df))
print(df["label"].value_counts().rename({0:"real",1:"fake"}))
df.sample(5, random_state=1)

## 3. Build the pipeline

Word n-grams capture topic/phrasing; character n-grams catch obfuscation. Logistic Regression stays interpretable.

In [ ]:
from src.train import build_pipeline   # reuse the exact production pipeline
from sklearn.model_selection import train_test_split

X, y = df["text"].astype(str), df["label"].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipe = build_pipeline()
print(pipe)

## 4. Cross-validation (robust estimate)

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
cv = StratifiedKFold(5, shuffle=True, random_state=42)
res = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring=["accuracy","f1"])
print(f"accuracy: {res['test_accuracy'].mean():.3f} ± {res['test_accuracy'].std():.3f}")
print(f"f1:       {res['test_f1'].mean():.3f} ± {res['test_f1'].std():.3f}")

## 5. Fit and evaluate on the held-out set

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, brier_score_loss, log_loss
pipe.fit(X_tr, y_tr)
proba = pipe.predict_proba(X_te)[:,1]
pred = (proba >= 0.5).astype(int)
print(classification_report(y_te, pred, target_names=["real","fake"]))
print(f"ROC-AUC : {roc_auc_score(y_te, proba):.3f}")
print(f"Brier   : {brier_score_loss(y_te, proba):.3f}  (lower = better calibrated)")
print(f"log-loss: {log_loss(y_te, proba, labels=[0,1]):.3f}")

## 6. Confusion matrix, ROC & calibration

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, pred), display_labels=["real","fake"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion matrix")

fpr, tpr, _ = roc_curve(y_te, proba)
ax[1].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_te, proba):.3f}"); ax[1].plot([0,1],[0,1],"--",c="gray")
ax[1].set_title("ROC curve"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].legend()

frac, mean = calibration_curve(y_te, proba, n_bins=10)
ax[2].plot(mean, frac, "o-", label="model"); ax[2].plot([0,1],[0,1],"--",c="gray",label="perfect")
ax[2].set_title("Calibration"); ax[2].set_xlabel("predicted"); ax[2].set_ylabel("observed"); ax[2].legend()
plt.tight_layout(); plt.show()

## 7. Most informative words

Logistic Regression coefficients power the 'why?' explanation in the app.

In [ ]:
import numpy as np
feats = pipe.named_steps["features"]; clf = pipe.named_steps["clf"]
names = feats.get_feature_names_out(); coefs = clf.coef_[0]
word = [(names[i][6:], coefs[i]) for i in range(len(names)) if names[i].startswith("word__")]
word.sort(key=lambda p: p[1])
print("Top REAL indicators:", [w for w,_ in word[:12]])
print("Top FAKE indicators:", [w for w,_ in word[-12:]][::-1])

## 8. End-to-end: verdict + signals + source reputation

This mirrors exactly what the app and CLI do.

In [ ]:
from src.predict import analyze
for text, url in [
    ("SHOCKING!! You won\'t BELIEVE the SECRET they are HIDING!!! Share before DELETED!!!", None),
    ("According to a report published by the ministry, an independent panel reviewed the data.", "https://reuters.com/x"),
    ("According to a report, officials reviewed the data before outlining next steps.", "https://worldnewsdailyreport.com/x"),
]:
    r = analyze(text, url=url)
    print(f"{r['verdict']:12s} score={r['credibility_score']:5.1f}  source={r['source']['tier']:9s}  {text[:45]}...")

## 9. Save the model

`src/train.py` does all of the above as a script and also writes the report plots to `reports/`.

In [ ]:
import joblib, os
os.makedirs("models", exist_ok=True)
joblib.dump(pipe, "models/fake_news_model.joblib")
print("saved -> models/fake_news_model.joblib")

---
### Next steps
- Swap in the real Kaggle dataset for publishable accuracy (auto-detected by `make_dataset.py`).
- The app (`streamlit run app.py`) adds URL fetching, batch scoring, and Claude summaries
  on top of this classifier.